<a href="https://colab.research.google.com/github/mostofa89/TrafficCongestionRiskZones./blob/main/TrafficCongestionRiskZones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

#Loading Datasets

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ucimachinelearning/vanet-traffic-congestion-dataset")

print("Path to dataset files:", path)

In [ ]:
print(os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "vanet_traffic_data.csv"))

print("DataFrame created successfully!")
print("Shape:", df.shape)

display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

#Data PreProcessing

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.columns.tolist()

In [ ]:
df.isnull().sum()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

In [ ]:
df.head()

In [ ]:
df.dtypes

##Feature Engineering

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

In [ ]:
df["timestamp_hour"] = df["timestamp"].dt.hour
df["timestamp_day_of_week"] = df["timestamp"].dt.dayofweek
df["timestamp_month"] = df["timestamp"].dt.month

df.head()

In [ ]:
df = df.drop("timestamp", axis=1)

df.head()

In [ ]:
print(df["timestamp_month"].value_counts())

In [ ]:
df = df.drop("timestamp_month", axis=1)

In [ ]:
le = LabelEncoder()

df["label_encoded"] = le.fit_transform(df["label"])

print("\nLabel mapping:")
for number, label in enumerate(le.classes_):
    print(number, "=", label)

df.drop("label", axis=1)

In [ ]:
print("Road segments:")
print(df["road_segment_id"].unique())

print("\nLabels:")
print(df["label"].unique())

In [ ]:
LabelEncoder().fit_transform(df["road_segment_id"])
df = df.drop("road_segment_id", axis=1)

In [ ]:
df.head()

In [ ]:
df.duplicated().sum()

## Graphical Analysis

In [ ]:
sns.countplot(data=df, x="label")
plt.title("Traffic Congestion Class Distribution")
plt.xlabel("Congestion Level")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.stripplot(
    data=df,
    x="label",
    y="avg_speed_kmph",
    jitter=True,
    alpha=0.5
)

plt.title("Average Speed vs Traffic Congestion Level")
plt.xlabel("Congestion Level")
plt.ylabel("Average Speed (km/h)")
plt.show()

In [ ]:
sns.stripplot(
    data=df,
    x="label",
    y="avg_speed_kmph",
    jitter=0.25,
    alpha=0.15,
    size=2
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="density_veh_per_km",
    y="avg_speed_kmph",
    hue="label",
    alpha=0.5
)

plt.title("Traffic Density vs Average Speed")
plt.xlabel("Vehicle Density (vehicles/km)")
plt.ylabel("Average Speed (km/h)")
plt.show()

## Correlation Analysis

In [ ]:
# Select numerical columns
numeric_df = df.select_dtypes(include="number")

# Calculate correlation
corr = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(18, 14))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Traffic Features")
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
  plt.figure(figsize=(8, 4))
  print()
  print(f"========================= polt of {col} =========================")
  print()
  plt.boxplot(x=df[col])
  plt.title(f"Boxplot of {col}")
  plt.xlabel(col)
  plt.show()

In [ ]:
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)

  IQR = Q3 - Q1
  lower_bound = Q3 - IQR * 1.5
  upper_bound = Q3 + IQR * 1.5
  print(f"\n{col}")
  print("Q1:", Q1)
  print("Q3:", Q3)
  print("IQR:", IQR)
  print("Lower Bound:", lower_bound)
  print("Upper Bound:", upper_bound)
  outliers = df[
    (df[col] < lower_bound) |
    (df[col] > upper_bound)
  ]

  print("Number of outliers:", len(outliers))


In [ ]:
features = [
    "avg_wait_time_s",
    "flow_veh_per_hr",
    "queue_length_veh",
    "avg_accel_ms2",
    "temp_c",
    "rssi_dbm",
    "acceleration_directionality",
    "congestion_pressure"
]

for col in features:
    plt.figure(figsize=(16, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.show()

In [ ]:
for col in features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution - {col}")
    plt.show()

In [ ]:
skewness = df[features].skew().sort_values(ascending=False)

print(skewness)

In [ ]:
import numpy as np

skewed_features = [
    "congestion_pressure",
    "avg_wait_time_s",
    "queue_length_veh"
]

for col in skewed_features:
  print(col, "minimum =", df[col].min())

for col in skewed_features:
  df[col] = np.log1p(df[col])

print(df[skewed_features].skew())

In [ ]:
for col in skewed_features:

    plt.figure(figsize=(8, 4))

    sns.histplot(
        data=df,
        x=col,
        kde=True
    )

    plt.title(f"Distribution after Log Transformation: {col}")
    plt.show()